# 08 — LLM Cluster Labeling

Find the best strategy for generating human-readable labels and summaries using a local LLM.

**Prerequisites:**
- Run `06_clustering.ipynb` first (produces `best_labels__5k.npy`)
- Ollama running locally with at least one model pulled:
  ```bash
  ollama serve
  ollama pull llama3.2  # or mistral, qwen2.5, gemma3, etc.
  ```

## Two decisions

1. **Context strategy** — which docs to feed the LLM as cluster representatives?
   - A: Centroid-nearest (most average examples)
   - B: TF-IDF top-n (most distinctive examples)
   - C: Hybrid (centroid for label, TF-IDF for summary)

2. **Prompt strategy** — two label prompt versions, one summary template

**Optional:** Set `USE_BERTOPIC_LABELS = True` to run the same comparison on BERTopic cluster labels.

## Setup

In [ ]:
import sys, json, time
import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
sys.path.insert(0, '..')

from utils import load_config, get_preprocessor, embedding_path, load_array, array_exists

cfg = load_config(sample_size=5_000, device="cpu")

# ── Which labels to use ───────────────────────────────────────────────────────
USE_BERTOPIC_LABELS = False   # True → load BERTopic cluster labels for comparison

# ── Model and embedding used in nb 04 / 06 ───────────────────────────────────
BEST_MODEL       = "all-mpnet-base-v2"   # update from nb 04
BEST_INSTRUCTION = "no_inst"              # update from nb 04

# ── Ollama ────────────────────────────────────────────────────────────────────
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2"   # change to whichever model you have pulled
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Load texts
preprocess = get_preprocessor("minimal")
texts = []
with open(cfg.data_path) as f:
    for line in f:
        obj = json.loads(line)
        texts.append(preprocess(obj["text"]))
        if len(texts) >= cfg.sample_size:
            break

# Load embeddings (needed for centroid strategy)
emb_path = embedding_path(cfg.cache_dir, BEST_MODEL, cfg.sample_size,
                           instruction=BEST_INSTRUCTION)
embeddings = load_array(emb_path)

# Load cluster labels
k = cfg.sample_size // 1000
if USE_BERTOPIC_LABELS:
    try:
        from bertopic import BERTopic
        bt = BERTopic.load(str(cfg.bertopic_dir / f"default__{k}k"))
        topics, _ = bt.transform(texts, embeddings=embeddings)
        labels = np.array(topics)
        print(f"Using BERTopic labels  ({len(set(labels)) - 1} topics)")
    except Exception as e:
        print(f"Could not load BERTopic model: {e}")
        USE_BERTOPIC_LABELS = False

if not USE_BERTOPIC_LABELS:
    labels = np.load(cfg.clustering_dir / f"best_labels__{k}k.npy")
    print(f"Using custom pipeline labels  ({len(set(labels)) - 1} clusters + noise)")

cluster_ids = sorted(c for c in set(labels) if c != -1)
print(f"Clusters to label: {len(cluster_ids)}")

In [ ]:
# Verify Ollama is running
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama running. Available models: {models}")
    if OLLAMA_MODEL not in models and not any(OLLAMA_MODEL in m for m in models):
        print(f"  ⚠  {OLLAMA_MODEL!r} not found. Run: ollama pull {OLLAMA_MODEL}")
except Exception as e:
    print(f"Ollama not running: {e}\nStart with: ollama serve")

## Context strategies

In [ ]:
def get_centroid_docs(cluster_id: int, n: int = 5) -> list[str]:
    """Return the n docs closest to the cluster centroid."""
    mask = labels == cluster_id
    cluster_embs = embeddings[mask]
    cluster_texts = [t for t, m in zip(texts, mask) if m]
    centroid = cluster_embs.mean(axis=0, keepdims=True)
    sims = cosine_similarity(centroid, cluster_embs)[0]
    top_idx = sims.argsort()[::-1][:n]
    return [cluster_texts[i] for i in top_idx]


def get_tfidf_docs(cluster_id: int, n: int = 5) -> list[str]:
    """Return the n most distinctive docs for the cluster (TF-IDF vs. rest)."""
    mask = labels == cluster_id
    cluster_texts = [t for t, m in zip(texts, mask) if m]
    other_texts   = [t for t, m in zip(texts, mask) if not m]

    tfidf = TfidfVectorizer(max_features=5_000, stop_words="english")
    tfidf_matrix = tfidf.fit_transform(cluster_texts + other_texts)
    scores = tfidf_matrix[:len(cluster_texts)].sum(axis=1).A1
    top_idx = scores.argsort()[::-1][:n]
    return [cluster_texts[i] for i in top_idx]


def format_docs(docs: list[str], max_chars: int = 300) -> str:
    return "\n\n".join(f"- {d[:max_chars]}" for d in docs)


# Preview
c0 = cluster_ids[0]
print(f"Cluster {c0} — centroid sample:")
print(get_centroid_docs(c0, n=1)[0][:200])
print(f"\nCluster {c0} — TF-IDF sample:")
print(get_tfidf_docs(c0, n=1)[0][:200])

## Prompt templates

In [ ]:
LABEL_PROMPT_V1 = """You are analyzing customer reviews. The following reviews all belong to the same topic cluster.

Reviews:
{docs}

Give a short, specific topic label (3-6 words) that describes what these reviews have in common.
Label only, no explanation."""

LABEL_PROMPT_V2 = """Analyze these customer reviews and identify their common theme.

Reviews:
{docs}

Respond with ONLY a topic label (3-6 words). Be specific, not generic."""

SUMMARY_PROMPT = """You are analyzing a cluster of customer reviews with the theme: \"{label}\".

Representative reviews:
{docs}

Write a 2-3 sentence summary of what customers in this cluster are saying.
Be specific and factual. Do not repeat the label."""


def ollama_generate(prompt: str, model: str = OLLAMA_MODEL, timeout: int = 60) -> str:
    payload = {"model": model, "prompt": prompt, "stream": False}
    r = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["response"].strip()

## Label generation experiment — first 10 clusters

In [ ]:
EVAL_CLUSTERS = cluster_ids[:10]

results = []
for cid in EVAL_CLUSTERS:
    centroid_docs = get_centroid_docs(cid, n=5)
    tfidf_docs    = get_tfidf_docs(cid, n=5)

    label_centroid_v1 = ollama_generate(LABEL_PROMPT_V1.format(docs=format_docs(centroid_docs)))
    label_centroid_v2 = ollama_generate(LABEL_PROMPT_V2.format(docs=format_docs(centroid_docs)))
    label_tfidf_v1    = ollama_generate(LABEL_PROMPT_V1.format(docs=format_docs(tfidf_docs)))

    results.append({
        "cluster": cid,
        "n_docs": int((labels == cid).sum()),
        "centroid_v1": label_centroid_v1,
        "centroid_v2": label_centroid_v2,
        "tfidf_v1": label_tfidf_v1,
    })
    print(f"Cluster {cid:>3d}: centroid_v1=[{label_centroid_v1}]  "
          f"centroid_v2=[{label_centroid_v2}]  tfidf_v1=[{label_tfidf_v1}]")

df_labels = pd.DataFrame(results)
df_labels

## Summary generation — best strategy

In [ ]:
# Update after reviewing df_labels above
BEST_STRATEGY     = "centroid"         # "centroid" | "tfidf"
BEST_LABEL_PROMPT = LABEL_PROMPT_V1

get_docs = get_centroid_docs if BEST_STRATEGY == "centroid" else get_tfidf_docs

print("Generating labels + summaries for first 5 clusters...\n")
for cid in EVAL_CLUSTERS[:5]:
    docs = get_docs(cid, n=5)
    label   = ollama_generate(BEST_LABEL_PROMPT.format(docs=format_docs(docs)))
    summary = ollama_generate(SUMMARY_PROMPT.format(label=label, docs=format_docs(docs)))
    print(f"── Cluster {cid} ({(labels == cid).sum()} docs) ──")
    print(f"Label  : {label}")
    print(f"Summary: {summary}")
    print()

## Human evaluation

Fill in scores (1–5) after reading the output above.
- 1 = wrong or too generic
- 5 = specific and correct

In [ ]:
eval_data = {
    "cluster":           EVAL_CLUSTERS,
    "centroid_v1_score": [None] * len(EVAL_CLUSTERS),   # fill in 1–5
    "centroid_v2_score": [None] * len(EVAL_CLUSTERS),
    "tfidf_v1_score":    [None] * len(EVAL_CLUSTERS),
    "notes":             [""]  * len(EVAL_CLUSTERS),
}
pd.DataFrame(eval_data)

## BERTopic keyword comparison (optional)

BERTopic generates c-TF-IDF keyword lists automatically — useful as a cross-check against LLM labels.
Only runs if `USE_BERTOPIC_LABELS = True` was set above.

In [ ]:
if USE_BERTOPIC_LABELS and 'bt' in dir():
    print("BERTopic keyword lists (top 6 words per topic):")
    for tid in cluster_ids[:10]:
        words = [w for w, _ in bt.get_topic(tid)[:6]]
        print(f"  Topic {tid:>3d}: {', '.join(words)}")
else:
    print("BERTopic keywords not shown (USE_BERTOPIC_LABELS=False or model not loaded).")

## Findings

**Context strategy winner:** `centroid` / `tfidf` / `hybrid`  
**Prompt version winner:** `v1` / `v2`  
**LLM model used:** _fill_  

Human evaluation summary:
- Average label quality centroid_v1: ___/5
- Average label quality centroid_v2: ___/5
- Average label quality tfidf_v1:   ___/5

**Production templates to use in the backend:**

```python
LABEL_PROMPT = """
# paste winner here
"""

SUMMARY_PROMPT = """
# paste winner here
"""
```

**n representative docs:** `___` centroid + `___` TF-IDF